## 1. Partindo de páginas recuperáveis

A ingestão não é o foco deste artigo. Vamos começar com páginas já preparadas como unidades recuperáveis. Cada registro tem texto e metadados suficientes para aparecer no resultado depois da busca.


In [1]:
records = [
    {
        "id": "Livro 1:p12",
        "title": "Arriscando a Própria Pele",
        "author": "Nassim Taleb",
        "page": 12,
        "text": "Risco e decisão mudam quando a pessoa sofre as consequências das próprias escolhas.",
    },
    {
        "id": "Livro 1:p41",
        "title": "Arriscando a Própria Pele",
        "author": "Nassim Taleb",
        "page": 41,
        "text": "Sistemas robustos precisam considerar assimetria, incentivos e exposição real ao erro.",
    },
    {
        "id": "Livro 2:p8",
        "title": "Antifrágil",
        "author": "Nassim Taleb",
        "page": 8,
        "text": "Alguns sistemas melhoram quando passam por volatilidade, pressão e incerteza.",
    },
    {
        "id": "Livro 2:p77",
        "title": "Antifrágil",
        "author": "Nassim Taleb",
        "page": 77,
        "text": "Decisão sob incerteza exige avaliar risco antes de aceitar perdas irreversíveis.",
    },
    {
        "id": "Livro 3:p23",
        "title": "Arquitetura de APIs",
        "author": "Equipe Técnica",
        "page": 23,
        "text": "Contratos de APIs definem formatos de requisição, resposta, autenticação e versionamento.",
    },
    {
        "id": "Livro 3:p67",
        "title": "Arquitetura de APIs",
        "author": "Equipe Técnica",
        "page": 67,
        "text": "Integrações confiáveis precisam de logs, timeouts, retentativas e monitoramento operacional.",
    },
]

print(f"registros recuperáveis: {len(records)}")

for record in records:
    print(f"{record['id']} | {record['title']} | p. {record['page']}")


registros recuperáveis: 6
Livro 1:p12 | Arriscando a Própria Pele | p. 12
Livro 1:p41 | Arriscando a Própria Pele | p. 41
Livro 2:p8 | Antifrágil | p. 8
Livro 2:p77 | Antifrágil | p. 77
Livro 3:p23 | Arquitetura de APIs | p. 23
Livro 3:p67 | Arquitetura de APIs | p. 67


## 2. Gerando embeddings dos textos

Agora cada página vira um vetor denso. O modelo usado aqui é pequeno e serve para demonstração. O ponto principal é observar a mudança de representação: o texto deixa de ser apenas uma lista de termos e passa a ser uma sequência de números comparável com outras sequências de números.


In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
texts = [record["text"] for record in records]
document_embeddings = model.encode(texts)

print(f"total de embeddings: {len(document_embeddings)}")
print(f"dimensões por embedding: {len(document_embeddings[0])}")
print("primeiros 8 números do primeiro embedding:")
print([round(float(value), 4) for value in document_embeddings[0][:8]])


total de embeddings: 6
dimensões por embedding: 384
primeiros 8 números do primeiro embedding:
[0.0287, 0.1036, -0.015, -0.0503, -0.1111, 0.036, 0.0405, 0.0089]


## 3. Vetorizando a query

A query precisa passar pelo mesmo modelo. Isso mantém query e documentos no mesmo espaço vetorial. A busca semântica compara esses vetores, não os textos diretamente.


In [3]:
query = "como decidir quando não sabemos o que vai acontecer"
query_embedding = model.encode([query])[0]

print(f"query: {query}")
print(f"dimensões do embedding da query: {len(query_embedding)}")
print("primeiros 8 números do embedding da query:")
print([round(float(value), 4) for value in query_embedding[:8]])


query: como decidir quando não sabemos o que vai acontecer
dimensões do embedding da query: 384
primeiros 8 números do embedding da query:
[-0.0033, 0.078, -0.0054, 0.0039, -0.061, 0.0071, 0.0595, -0.0121]


## 4. Calculando cosine similarity

A cosine similarity mede o quanto dois vetores apontam para direções parecidas. Para evitar que o score pareça mágico, vamos calcular explicitamente a similaridade entre a query e uma das páginas.


In [4]:
import numpy as np

def cosine_similarity(vector_a, vector_b):
    dot_product = np.dot(vector_a, vector_b)
    norm_a = np.linalg.norm(vector_a)
    norm_b = np.linalg.norm(vector_b)
    return dot_product / (norm_a * norm_b)

example_record = records[0]
example_embedding = document_embeddings[0]
score = cosine_similarity(query_embedding, example_embedding)

print(f"query: {query}")
print("comparando com:")
print(f"{example_record['title']} | p. {example_record['page']}")
print(example_record["text"])
print(f"score de similaridade: {score:.4f}")


query: como decidir quando não sabemos o que vai acontecer
comparando com:
Arriscando a Própria Pele | p. 12
Risco e decisão mudam quando a pessoa sofre as consequências das próprias escolhas.
score de similaridade: 0.6002


## 5. Gerando o ranking semântico

Com a mesma função, calculamos a similaridade da query contra todos os registros e ordenamos do mais próximo para o menos próximo. Aqui `top_k = 3` define quantos candidatos aparecem no retorno.


In [5]:
def semantic_search(query, records, document_embeddings, top_k=3):
    query_embedding = model.encode([query])[0]
    ranking = []

    for record, document_embedding in zip(records, document_embeddings):
        score = cosine_similarity(query_embedding, document_embedding)
        ranking.append((float(score), record))

    return sorted(ranking, key=lambda item: item[0], reverse=True)[:top_k]

results = semantic_search(query, records, document_embeddings, top_k=3)

for position, (score, record) in enumerate(results, start=1):
    print(f"#{position} {record['id']} | score={score:.4f}")
    print(f"{record['title']} | p. {record['page']}")
    print(record["text"])
    print()


#1 Livro 1:p12 | score=0.6002
Arriscando a Própria Pele | p. 12
Risco e decisão mudam quando a pessoa sofre as consequências das próprias escolhas.

#2 Livro 2:p8 | score=0.5966
Antifrágil | p. 8
Alguns sistemas melhoram quando passam por volatilidade, pressão e incerteza.

#3 Livro 2:p77 | score=0.5202
Antifrágil | p. 77
Decisão sob incerteza exige avaliar risco antes de aceitar perdas irreversíveis.



## 6. Lendo o resultado

A query não pergunta por `risco`, `volatilidade` ou `incerteza` literalmente. Mesmo assim, as páginas sobre decisão, consequências, volatilidade e incerteza aparecem acima das páginas sobre APIs. Esse é o sinal que embeddings adicionam ao retrieval: proximidade de significado.

Isso não transforma o resultado em verdade absoluta. O score continua sendo um sinal. Ele ajuda a ordenar candidatos, mas a qualidade final ainda depende do texto representado, do modelo, do `top_k` e do tipo de pergunta feita pelo usuário.
